In [1]:
# Cell 1: Setup
from pydantic_ai import Tool
from pydantic_ai.hooks.agent import ReactiveAgent

In [2]:
# Cell 2: Create a Simple Tool
async def greet(name: str) -> str:
    """Greet someone."""
    return f"Hello, {name}!"

tool = Tool(
    name="greet",
    function=greet,
    description="Greets a person"
)

In [3]:
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="env/.env")

# Create the model with OpenRouter configuration
api_key = os.environ.get("OPENROUTER_API_KEY")
base_url = os.environ.get("OPENROUTER_BASE_URL")

# Update to match the template structure
model = OpenAIModel(
    model_name="gpt-3.5-turbo",
    provider=OpenAIProvider(base_url=base_url, api_key=api_key),
)

In [4]:
# Cell 3: Create ReactiveAgent
agent = ReactiveAgent(
    model=model,
    tools=[tool],
    instructions="You are German and concise."
)

In [5]:
# Cell 4: Watch Model Changes
def on_model_change(change):
    print(f"Model changed from {change.old} to {change.new}")

# Watch the model value changes
agent.on.model.observe(on_model_change, names='value')

In [6]:
# Cell 5: Watch Instruction Changes
def on_instructions_change(change):
    print(f"Instructions changed from {change.old} to {change.new}")

# Watch the instructions value changes
agent.on.instructions.observe(on_instructions_change, names='value')

In [7]:
# Cell 6: Add Lifecycle Hooks
async def on_start(context):
    print(f"Agent starting with args: {context['args']}")

async def on_end(context):
    print("Agent finished")

# Current hook assignment
agent.on.start = on_start
agent.on.end = on_end

In [8]:
# Cell 7: Add Global Tool Hooks
async def before_any_tool(context):
    name = context['name']
    args = context['args']
    print(f"About to run tool: {name} with args: {args}")

async def after_any_tool(context):
    name = context['name']
    result = context['result']
    print(f"Tool {name} completed with result: {result}")

# Set the hooks
print("Before hook:", agent.on.before_any_tool)
agent.on.before_any_tool = before_any_tool
print("After setting before hook:", agent.on.before_any_tool)

print("\nAfter hook:", agent.on.after_any_tool)
agent.on.after_any_tool = after_any_tool
print("After setting after hook:", agent.on.after_any_tool)

Before hook: None
After setting before hook: <function before_any_tool at 0x1318ccc20>

After hook: None
After setting after hook: <function after_any_tool at 0x1318ccea0>


In [9]:
# Direct tool call
print("Direct tool call:")
await agent.call_tool(tool, "Alice")

Direct tool call:
About to run tool: greet with args: ('Alice',)
Tool greet completed with result: Hello, Alice!


'Hello, Alice!'

In [10]:
print("\nAgent run:")
await agent.run("Please greet Alice")


Agent run:
Agent starting with args: ('Please greet Alice',)
Node type: UserPromptNode
Node type: ModelRequestNode
Node type: CallToolsNode
Found CallToolsNode, response parts: ['ToolCallPart']
Processing part: ToolCallPart
About to run tool: greet with args: {"name":"Alice"}
Tool greet completed with result: Hello, Alice!
Node type: ModelRequestNode
Node type: CallToolsNode
Found CallToolsNode, response parts: ['TextPart']
Processing part: TextPart
Node type: End
Agent finished


AgentRunResult(output='If you need anything else, feel free to ask.')

In [11]:
# Cell 9: Change Model
new_model = OpenAIModel(
    model_name="gpt-4",
    provider=OpenAIProvider(base_url=base_url, api_key=api_key),
)

# This will trigger the model change observer
agent.model = new_model

Model changed from OpenAIModel() to OpenAIModel()


In [12]:
# Cell 10: Change Instructions
# This will trigger the instructions change observer
agent.instructions = "Be very funny."

Instructions changed from You are German and concise. to Be very funny.


In [13]:
print("\nAgent run:")
await agent.run("Please greet Bob")


Agent run:
Agent starting with args: ('Please greet Bob',)
Node type: UserPromptNode
Node type: ModelRequestNode
Node type: CallToolsNode
Found CallToolsNode, response parts: ['ToolCallPart']
Processing part: ToolCallPart
About to run tool: greet with args: {
"name": "Bob"
}
Tool greet completed with result: Hello, Bob!
Node type: ModelRequestNode
Node type: CallToolsNode
Found CallToolsNode, response parts: ['TextPart']
Processing part: TextPart
Node type: End
Agent finished


AgentRunResult(output="Why don't they play poker in the jungle? Because there are too many cheetahs! 🐆")

In [14]:
agent.on.model

In [15]:
agent.on.instructions.value

'Be very funny.'

In [16]:
# Watch for hook changes
def on_hook_change(change):
    print(f"Hook changed from {change.old} to {change.new}")

# Watch the hook changes
agent.on.model.observe(on_hook_change, names='hook')

# Test it by setting a new hook
async def new_hook(context):
    print("New hook called!")

agent.on.model.hook = new_hook


Hook changed from None to <function new_hook at 0x130ad32e0>


In [17]:
# Watch both hook and value changes
def on_any_change(change):
    print(f"Trait {change.name} changed from {change.old} to {change.new}")

# Watch both hook and value changes
agent.on.model.observe(on_any_change, names=['hook', 'value'])


In [18]:
# Create a ReactiveTool
from pydantic_ai.hooks.tool import ReactiveTool

async def multiply(x: int) -> int:
    """Multiply a number by itself."""
    return x * x

reactive_tool = ReactiveTool(
    name="multiply", 
    function=multiply,
    description="Multiplies a number by itself"
)

# Watch tool state changes
def watch_state(change):
    print(f"Tool state changed from {change.old} to {change.new}")

reactive_tool.state.observe(watch_state, names='state')

# Add tool hooks
async def log_before(context):
    print(f"Tool about to run with args: {context['args']}")

async def log_after(context):
    print(f"Tool got result: {context['result']}")

async def log_error(context):
    print(f"Tool error occurred: {context['error']}")

reactive_tool.on.before = log_before
reactive_tool.on.after = log_after
reactive_tool.on.error = log_error

# Create a new agent with the reactive tool
reactive_agent = ReactiveAgent(
    model=model,
    tools=[reactive_tool],
    instructions="You are a math tutor."
)

# Add agent hooks
async def agent_before_tool(context):
    name = context['name']
    args = context['args']
    print(f"Agent about to run tool: {name} with args: {args}")

async def agent_after_tool(context):
    name = context['name']
    result = context['result']
    print(f"Agent tool {name} completed with result: {result}")

reactive_agent.on.before_any_tool = agent_before_tool
reactive_agent.on.after_any_tool = agent_after_tool

# Test direct tool call
print("\nDirect tool call:")
await reactive_agent.call_tool(reactive_tool, 5)

# Test agent run
print("\nAgent run:")
await reactive_agent.run("What is 7 multiplied by itself?")



Direct tool call:
Agent about to run tool: multiply with args: (5,)
Agent tool multiply completed with result: 25

Agent run:
Node type: UserPromptNode
Node type: ModelRequestNode
Node type: CallToolsNode
Found CallToolsNode, response parts: ['ToolCallPart']
Processing part: ToolCallPart
Agent about to run tool: multiply with args: {"x":7}
Agent tool multiply completed with result: 49
Node type: ModelRequestNode
Node type: CallToolsNode
Found CallToolsNode, response parts: ['TextPart']
Processing part: TextPart
Node type: End


AgentRunResult(output='7 multiplied by itself is 49.')

In [19]:
# Test error handling
print("\nTesting error handling:")
try:
    await reactive_agent.call_tool(reactive_tool, "not a number")
except TypeError as e:
    print("Caught expected error")

# Check tool state
print("\nTool state:")
print(f"Current state: {reactive_tool.state.state}")
print(f"Last result: {reactive_tool.state.result}")
print(f"Last error: {reactive_tool.state.error}")
print(f"Last context: {reactive_tool.state.context}")



Testing error handling:
Agent about to run tool: multiply with args: ('not a number',)
Caught expected error

Tool state:
Current state: ToolState.IDLE
Last result: None
Last error: None
Last context: {}
